In [1]:
%pip install -q pypdf sentence-transformers gradio

In [2]:
import gradio as gr
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

chunks = []
vectors = None


# Read the PDF and prepare notes for searching
def upload_pdf(file):
    global chunks, vectors

    if file is None:
        return "Please upload a PDF first."

    try:
        reader = PdfReader(file)
        text = " ".join(page.extract_text() or "" for page in reader.pages)

        if not text.strip():
            return "No readable text found in this PDF."

        words = text.split()
        chunks = [
            " ".join(words[i:i + 100])
            for i in range(0, len(words), 80)
        ]

        vectors = model.encode(
            chunks, normalize_embeddings=True
        )

        return "PDF processed successfully! Start studying."

    except Exception as e:
        chunks = []
        vectors = None
        return f"Could not read PDF: {e}"


# Retrieve relevant notes using RAG
def search_notes(question):
    if vectors is None:
        return []

    question_vector = model.encode(
        [question], normalize_embeddings=True
    )[0]

    scores = np.dot(vectors, question_vector)
    best_ids = np.argsort(scores)[::-1][:3]

    return [chunks[i] for i in best_ids]


# Answer questions using retrieved notes
def ask_question(question):
    if not question.strip():
        return "Please enter a question."

    results = search_notes(question)

    if not results:
        return "Please upload and process a PDF first."

    return "\n\n".join(results)


# Create a simple summary from the notes
def summarize():
    if not chunks:
        return "Please upload and process a PDF first."

    text = " ".join(chunks[:3])
    sentences = text.replace("\n", " ").split(". ")

    return ". ".join(sentences[:5])


# Create a basic practice quiz from the notes
def make_quiz():
    if not chunks:
        return "Please upload and process a PDF first."

    sentences = [
        s.strip()
        for s in ". ".join(chunks[:3]).split(".")
        if len(s.split()) >= 6
    ]

    if not sentences:
        return "Not enough text to create a quiz."

    return "\n".join(
        f"Q{i + 1}. Explain this point in your own words:\n{s}."
        for i, s in enumerate(sentences[:5])
    )


# Build the interface
with gr.Blocks() as app:
    gr.Markdown("# AI Study Buddy")
    gr.Markdown("Upload your notes and use RAG to study.")

    pdf = gr.File(label="Upload PDF", file_types=[".pdf"])
    upload_btn = gr.Button("Process PDF")
    status = gr.Textbox(label="Status")
    upload_btn.click(upload_pdf, inputs=pdf, outputs=status)

    gr.Markdown("### Ask Questions")
    question = gr.Textbox(label="Enter your question")
    answer = gr.Textbox(label="Relevant answers from your notes", lines=8)
    gr.Button("Search Notes").click(
        ask_question, inputs=question, outputs=answer
    )

    gr.Markdown("### Summarize Notes")
    summary = gr.Textbox(label="Summary", lines=5)
    gr.Button("Create Summary").click(
        summarize, inputs=None, outputs=summary
    )

    gr.Markdown("### Practice Quiz")
    quiz = gr.Textbox(label="Quiz", lines=8)
    gr.Button("Create Quiz").click(
        make_quiz, inputs=None, outputs=quiz
    )

app.launch()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ee46c32f0399b3204.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
